# Plots Climaticos 16:9 — Juazeiro, BA
Formato **TikTok / Reels**  ·  Alta qualidade  ·  @reinanbr

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import matplotlib.patheffects as pe
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from datetime import datetime
from IPython.display import display, Image, clear_output
import ipywidgets as widgets
import warnings, io
warnings.filterwarnings('ignore')
print('imports OK')


imports OK


In [7]:
def to_dt(v):
    ts = (v - np.datetime64('1970-01-01T00:00')) / np.timedelta64(1, 's')
    return datetime.utcfromtimestamp(float(ts))

VARS_NEEDED = ['prmsl', 'prate', 'r2', 'cwat', 't2m']

try:
    from noawclg.main import get_noaa_data
    print('Baixando GFS...')
    _raw_full, _time_full, _lat, _lon = {}, None, None, None
    noaa = get_noaa_data(keys=VARS_NEEDED, date='20260405')
    view = noaa.get_data_from_place('Juazeiro, Bahia, Brazil')
    ds   = view.dataset
    for key in VARS_NEEDED:
        if _time_full is None:
            _time_full = ds['time'].values
            _lat = float(ds['latitude'].values)
            _lon = float(ds['longitude'].values)
        _raw_full[key] = ds[key].values.astype(float)
    if _raw_full['t2m'].mean() > 100:
        _raw_full['t2m'] -= 273.15
    DATA_SOURCE = 'GFS 0.25 / NOAA - NOMADS'
    print('GFS OK')
except Exception as e:
    print(f'Sintetico ({e})')
    _N = 24*7+1
    _time_full = np.array([
        np.datetime64('2026-04-04T00:00') + np.timedelta64(i, 'h')
        for i in range(_N)])
    np.random.seed(7)
    _h = np.arange(_N)
    _raw_full = {
        'prmsl': 1010 + 3*np.sin(_h/36*np.pi) + np.random.normal(0,.4,_N),
        'prate': np.clip(np.where(_h%48<6, np.random.exponential(1e-4,_N),0)
                         + np.random.exponential(5e-5,_N)*.2, 0, None),
        'r2':    np.clip(55+25*np.sin((_h%24-4)*np.pi/12)+np.random.normal(0,3,_N),15,98),
        'cwat':  np.clip(.15+.25*np.abs(np.sin(_h/48*np.pi))+np.random.exponential(.04,_N),0,None),
        't2m':   np.clip(30+8*np.sin((_h%24-6)*np.pi/12)+np.random.normal(0,.8,_N),18,42),
    }
    _lat, _lon = -9.4168, -40.5019
    DATA_SOURCE = 'Dados sinteticos (exemplo)'
print('Dados prontos')


Baixando GFS...
Sintetico ('GFSDatasetManager' object has no attribute 'build_multi_variable_dataset')
Dados prontos


In [8]:
BG       = '#07080C'
PANEL_BG = '#0C0E14'
CHART_BG = '#0A0B10'
TEXT_PRI = '#F0EDE8'
TEXT_SEC = '#7A7F8E'
TEXT_DIM = '#2E3140'
HANDLE   = '#FF5722'
GRID_C   = '#ffffff09'

VAR_CFG = {
    't2m': dict(
        label='TEMPERATURA', sublabel='2 metros acima do solo',
        unit='C', ylabel='Temperatura (C)', fmt='%.1f',
        refband=(20,30), refband_label='Conforto Termico',
        emoji='\U0001f321', bar_plot=False,
        colors=['#0D1F3C','#1565C0','#1E88E5','#FF8F00','#E53935','#7B1FA2'],
        accent='#FF8F00', accent_hi='#FF3D00',
    ),
    'prate': dict(
        label='PRECIPITACAO', sublabel='Taxa instantanea de chuva',
        unit='mm/h', ylabel='Taxa (mm/h)', fmt='%.3f',
        refband=None, refband_label=None,
        emoji='\U0001f327', bar_plot=True,
        colors=['#0D1B2E','#01579B','#0288D1','#4FC3F7','#B3E5FC'],
        accent='#29B6F6', accent_hi='#B3E5FC',
    ),
    'r2': dict(
        label='HUMIDADE RELATIVA', sublabel='Humidade do ar a 2 m',
        unit='%', ylabel='Humidade (%)', fmt='%.0f',
        refband=(60,80), refband_label='Faixa Confortavel',
        emoji='\U0001f4a7', bar_plot=False,
        colors=['#3E2000','#F57F17','#F9A825','#66BB6A','#00ACC1'],
        accent='#66BB6A', accent_hi='#00E5FF',
    ),
    'prmsl': dict(
        label='PRESSAO ATMOSFERICA', sublabel='Pressao ao nivel do mar',
        unit='hPa', ylabel='Pressao (hPa)', fmt='%.1f',
        refband=None, refband_label=None,
        emoji='\U0001f32c', bar_plot=False,
        colors=['#1A0533','#4A148C','#7B1FA2','#7986CB','#B3C5EF'],
        accent='#7986CB', accent_hi='#C5CAE9',
    ),
    'cwat': dict(
        label='AGUA EM NUVENS', sublabel='Agua liquida total na coluna',
        unit='kg/m2', ylabel='Agua em nuvens (kg/m2)', fmt='%.3f',
        refband=None, refband_label=None,
        emoji='\u2601', bar_plot=False,
        colors=['#0D1117','#1C2333','#37474F','#78909C','#CFD8DC'],
        accent='#90A4AE', accent_hi='#ECEFF1',
    ),
}
for k, v in VAR_CFG.items():
    v['cmap'] = LinearSegmentedColormap.from_list(k, v['colors'], N=512)
print('Paleta OK')


Paleta OK


In [9]:
def _prepare(periodo):
    STEP_H = 1 if periodo == '1dia' else 3
    N_H    = 24 if periodo == '1dia' else 24*7
    idxs   = np.arange(0, N_H+1, STEP_H)
    idxs   = idxs[idxs < len(_time_full)]
    tv     = _time_full[idxs]
    raw    = {k: _raw_full[k][idxs].copy() for k in VARS_NEEDED}
    raw['prate'] *= 3600.0
    dts    = [to_dt(v) for v in tv]
    tx     = np.array([(d-dts[0]).total_seconds()/86400 for d in dts])
    return raw, dts, tx

def _ticks(dts, tx, periodo):
    if periodo == '1dia':
        idxs = [i for i,d in enumerate(dts) if d.hour % 3 == 0]
        return [tx[i] for i in idxs], [dts[i].strftime('%Hh') for i in idxs]
    unique = sorted(set(d.date() for d in dts))
    tdts   = [dts[next(i for i,dd in enumerate(dts) if dd.date()==u)] for u in unique]
    return [(d-dts[0]).total_seconds()/86400 for d in tdts], [d.strftime('%d/%m') for d in tdts]

def _colored_line(ax, tx, vals, cmap, lw=3.0):
    lo, hi = vals.min(), vals.max()
    pts  = np.array([tx, vals]).T.reshape(-1,1,2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    norm = Normalize(lo, hi)
    for width, alpha in [(lw, 1.0), (lw*4.5, 0.10)]:
        lc = LineCollection(segs, cmap=cmap, norm=norm,
                            linewidth=width, alpha=alpha, zorder=6 if width==lw else 5,
                            capstyle='round')
        lc.set_array((vals[:-1]+vals[1:])/2)
        ax.add_collection(lc)
    ti = np.linspace(tx[0], tx[-1], 800)
    ci = np.interp(ti, tx, vals)
    nv = (ci-lo)/max(hi-lo, 1e-9)
    for k in range(0, len(ti)-1, 2):
        ax.fill_between(ti[k:k+2], lo-abs(hi-lo)*.05, ci[k:k+2],
                        color=cmap(nv[k]), alpha=0.09, linewidth=0, zorder=2)

def _mark_ext(ax, tx, vals, accent, accent_hi, fmt):
    lo, hi = float(vals.min()), float(vals.max())
    span   = hi - lo
    x_end  = tx[-1]
    def side(i): return 0.03*x_end if tx[i] < x_end*0.72 else -0.18*x_end
    for idx, col, tag, dy in [
            (int(np.argmax(vals)), accent_hi, 'MAX', span*.12),
            (int(np.argmin(vals)), accent,    'MIN', -span*.15)]:
        ax.scatter(tx[idx], vals[idx], s=110, color=col, zorder=12,
                   edgecolors='white', linewidths=1.4)
        ax.annotate(f'{tag}  {vals[idx]:{fmt[1:]}}',
                    xy=(tx[idx], vals[idx]),
                    xytext=(tx[idx]+side(idx), vals[idx]+dy),
                    color=col, fontsize=9.5, fontweight='bold',
                    fontfamily='monospace', zorder=13, va='center',
                    arrowprops=dict(arrowstyle='-', color=col+'99', lw=1.0),
                    path_effects=[pe.withStroke(linewidth=3, foreground='#07080C')])

def _draw_bg(fig, accent):
    ax0 = fig.add_axes([0,0,1,1])
    ax0.set_xlim(0,1); ax0.set_ylim(0,1); ax0.axis('off')
    ax0.patch.set_facecolor(BG)
    xs = np.linspace(.02,.98,38); ys = np.linspace(.02,.98,22)
    for x in xs:
        for y in ys:
            ax0.plot(x, y, '.', color=accent, alpha=0.022, markersize=2, zorder=0)
    rect = Rectangle((0,0), .265, 1., transform=fig.transFigure,
                      color=PANEL_BG, zorder=1, clip_on=False)
    fig.add_artist(rect)
    for (x0,x1),(y0,y1),lw,alpha in [
            ([.265,.265],[0,1],2.,.65),
            ([0,1],[.977,.977],4.,.90),
            ([0,1],[.018,.018],1.5,.45)]:
        fig.add_artist(mlines.Line2D([x0,x1],[y0,y1], transform=fig.transFigure,
                                      color=accent, linewidth=lw, alpha=alpha, zorder=2))

def _sep(fig, y, accent):
    fig.add_artist(mlines.Line2D([.025,.250],[y,y], transform=fig.transFigure,
                                  color=accent+'55', linewidth=1, zorder=3))

def _draw_left(fig, cfg, dts, periodo, run_str):
    vals=cfg['_vals']; accent=cfg['accent']; acc_hi=cfg['accent_hi']
    unit=cfg['unit']; ps='24 HORAS' if periodo=='1dia' else '7 DIAS'
    T=lambda x,y,s,**kw: fig.text(x,y,s,**kw)
    T(.025,.895, cfg['emoji'], fontsize=52, va='top', zorder=3, color=accent)
    T(.025,.820, cfg['label'], color=accent, fontsize=10.5, fontweight='black',
      fontfamily='monospace', va='top', zorder=3)
    T(.025,.782, cfg['sublabel'], color=TEXT_SEC, fontsize=7.5,
      fontfamily='monospace', va='top', zorder=3)
    _sep(fig,.764,accent)
    T(.025,.748,'LOCALIZACAO',color=TEXT_DIM+'EE',fontsize=6.5,
      fontfamily='monospace',fontweight='bold',va='top',zorder=3)
    T(.025,.720,'Juazeiro - BA',color=TEXT_PRI,fontsize=13.5,
      fontweight='black',va='top',zorder=3)
    T(.025,.693,f'{abs(_lat):.3f}S  {abs(_lon):.3f}W',
      color=TEXT_SEC,fontsize=7.5,fontfamily='monospace',va='top',zorder=3)
    T(.025,.657,'PERIODO',color=TEXT_DIM+'EE',fontsize=6.5,
      fontfamily='monospace',fontweight='bold',va='top',zorder=3)
    T(.025,.629,ps,color=accent,fontsize=12.5,fontweight='black',
      fontfamily='monospace',va='top',zorder=3)
    T(.025,.604,f"{dts[0].strftime('%d/%m')} -> {dts[-1].strftime('%d/%m/%Y')}",
      color=TEXT_SEC,fontsize=7.5,fontfamily='monospace',va='top',zorder=3)
    _sep(fig,.587,accent)
    stats=[('MAXIMA',f'{vals.max():.3g}',unit,acc_hi),
           ('MINIMA',f'{vals.min():.3g}',unit,accent),
           ('MEDIA', f'{vals.mean():.3g}',unit,TEXT_PRI),
           ('DESVIO',f'{vals.std():.3g}', unit,TEXT_SEC),
           ('AMPLITUDE',f'{vals.max()-vals.min():.3g}',unit,'#6C7A95')]
    y0=.570
    for lbl,val,u,col in stats:
        T(.025,y0,lbl,color=TEXT_SEC,fontsize=6,fontfamily='monospace',fontweight='bold',va='top',zorder=3)
        T(.025,y0-.030,f'{val} {u}',color=col,fontsize=11.5,fontweight='black',va='top',zorder=3)
        fig.add_artist(mlines.Line2D([.025,.225],[y0-.069,y0-.069],
                                      transform=fig.transFigure,color=col,linewidth=2,alpha=.55,zorder=3))
        y0-=.084
    fig.add_artist(mlines.Line2D([.025,.250],[.150,.150],transform=fig.transFigure,
                                  color=accent+'33',linewidth=.8,zorder=3))
    T(.025,.138,'FONTE',color=TEXT_DIM+'EE',fontsize=6,fontfamily='monospace',fontweight='bold',va='top',zorder=3)
    T(.025,.118,DATA_SOURCE,color=TEXT_SEC,fontsize=6.5,fontfamily='monospace',va='top',zorder=3)
    T(.025,.093,f'GFS 0.25  |  {run_str}',color=TEXT_DIM+'CC',fontsize=6,fontfamily='monospace',va='top',zorder=3)
    T(.025,.040,'@reinanbr_',color=HANDLE,fontsize=17,fontweight='black',fontfamily='monospace',va='top',zorder=3,
      path_effects=[pe.withStroke(linewidth=6,foreground=PANEL_BG)])

def _draw_chart(fig, cfg, tx, vals, tick_x, tick_lbl, periodo):
    accent=cfg['accent']; acc_hi=cfg['accent_hi']; cmap=cfg['cmap']
    lo=float(vals.min()); hi=float(vals.max()); span=hi-lo
    ps_lbl='24 h' if periodo=='1dia' else '7 dias'
    T=lambda x,y,s,**kw: fig.text(x,y,s,**kw)
    T(.283,.936,f'{cfg["label"]}  -  {ps_lbl}',color=TEXT_PRI,fontsize=14.5,
      fontweight='black',va='top',zorder=3,
      path_effects=[pe.withStroke(linewidth=5,foreground=BG)])
    T(.283,.899,cfg['sublabel'],color=TEXT_SEC,fontsize=8.5,fontfamily='monospace',va='top',zorder=3)
    ax=fig.add_axes([.283,.155,.700,.722])
    ax.set_facecolor(CHART_BG)
    ax.set_xlim(tx[0],tx[-1])
    ax.set_ylim(lo-span*.14,hi+span*.22)
    step=max(span/6,1e-9); mag=10**np.floor(np.log10(step)); step=round(step/mag)*mag
    ys=np.arange(np.floor(lo/step)*step,np.ceil(hi/step)*step+step,step)
    for i,y in enumerate(ys):
        if i%2==0: ax.axhspan(y,y+step,color='#FFFFFF06',zorder=0,linewidth=0)
        ax.axhline(y,color=GRID_C,linewidth=.8,zorder=1)
    if cfg.get('refband'):
        r0,r1=cfg['refband']
        ax.axhspan(r0,r1,color=accent+'18',zorder=1,linewidth=0)
        ax.annotate(f' {cfg["refband_label"]}',xy=(tx[-1],(r0+r1)/2),
                    color=accent+'99',fontsize=8,fontfamily='monospace',
                    va='center',ha='right',zorder=4)
    if cfg.get('bar_plot'):
        norm_v=(vals-lo)/max(hi-lo,1e-9)
        colors=[cmap(float(v)) for v in norm_v]
        bw=(tx[1]-tx[0])*.82
        ax.bar(tx,vals,width=bw,color=colors,alpha=.88,zorder=4,linewidth=0)
        ax.bar(tx,vals,width=bw*1.6,color=accent,alpha=.055,zorder=3,linewidth=0)
        mv=float(vals.mean())
        ax.axhline(mv,color=accent,linewidth=1.6,linestyle='--',alpha=.75,zorder=6,
                   label=f'Media  {mv:.3g} {cfg["unit"]}')
        ax.legend(loc='upper right',framealpha=0,labelcolor=TEXT_SEC,fontsize=8.5)
    else:
        _colored_line(ax,tx,vals,cmap)
        _mark_ext(ax,tx,vals,accent,acc_hi,cfg['fmt'])
    ax.set_xticks(tick_x)
    ax.set_xticklabels(tick_lbl,color=TEXT_SEC,fontsize=9.5,fontfamily='monospace')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter(cfg['fmt']))
    ax.tick_params(axis='y',colors=TEXT_SEC,labelsize=9.5,length=4,pad=7)
    ax.tick_params(axis='x',colors=TEXT_SEC,length=5,pad=5)
    for sp in ax.spines.values(): sp.set_edgecolor(accent+'55'); sp.set_linewidth(1.4)
    ax.set_ylabel(cfg['ylabel'],color=TEXT_SEC,fontsize=9,fontfamily='monospace',labelpad=12)
    cb=fig.add_axes([.283,.120,.700,.018])
    grad=np.linspace(0,1,512).reshape(1,-1)
    cb.imshow(grad,aspect='auto',cmap=cmap,origin='lower'); cb.axis('off')
    T(.283,.104,f'{lo:{cfg["fmt"][1:]}} {cfg["unit"]}',color=TEXT_SEC,fontsize=7.5,
      fontfamily='monospace',ha='left',va='top',zorder=3)
    T(.983,.104,f'{hi:{cfg["fmt"][1:]}} {cfg["unit"]}',color=TEXT_SEC,fontsize=7.5,
      fontfamily='monospace',ha='right',va='top',zorder=3)

print('Helpers OK')


Helpers OK


In [10]:
def gerar_plots(periodo, vars_sel, salvar, dpi_val):
    raw, dts, tx = _prepare(periodo)
    tick_x, tick_lbl = _ticks(dts, tx, periodo)
    run_str = datetime.utcnow().strftime('%Y-%m-%dT%H:%MZ')
    suffix  = '1d' if periodo == '1dia' else '7d'
    KEY_MAP = {'prmsl':raw['prmsl'],'prate':raw['prate'],
               'r2':raw['r2'],'cwat':raw['cwat'],'t2m':raw['t2m']}

    for key in vars_sel:
        cfg = dict(VAR_CFG[key])
        cfg['_vals'] = KEY_MAP[key]
        vals = cfg['_vals']

        # 16:9 exato — usa Figure diretamente (sem plt) para compatibilidade total
        fig = matplotlib.figure.Figure(figsize=(19.2, 10.8), dpi=dpi_val)
        FigureCanvasAgg(fig)
        fig.patch.set_facecolor(BG)

        _draw_bg(fig, cfg['accent'])
        _draw_left(fig, cfg, dts, periodo, run_str)
        _draw_chart(fig, cfg, tx, vals, tick_x, tick_lbl, periodo)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=dpi_val,
                    bbox_inches='tight', facecolor=BG, edgecolor='none')
        buf.seek(0)
        display(Image(data=buf.read()))

        if salvar:
            fname = f'juazeiro_{key}_{suffix}.png'
            buf.seek(0)
            with open(fname, 'wb') as f:
                f.write(buf.read())
            print(f'Salvo: {fname}  ({dpi_val} dpi  |  19.2x10.8 pol = 16:9)')

print('gerar_plots() OK')


gerar_plots() OK


In [11]:
w_periodo = widgets.ToggleButtons(
    options=[('1 Dia (24h)', '1dia'), ('1 Semana (7 dias)', '1semana')],
    value='1semana', description='Periodo:',
    style={'button_width':'170px','description_width':'70px'},
    layout=widgets.Layout(margin='6px 0'),
)
w_vars = widgets.SelectMultiple(
    options=[('Temperatura (t2m)','t2m'),('Precipitacao (prate)','prate'),
             ('Humidade (r2)','r2'),('Pressao (prmsl)','prmsl'),
             ('Nuvens (cwat)','cwat')],
    value=['t2m','prate','r2','prmsl','cwat'],
    description='Variaveis:', rows=5,
    layout=widgets.Layout(width='340px'),
    style={'description_width':'80px'},
)
w_dpi = widgets.RadioButtons(
    options=[('Previa rapida (72 dpi)',72),
             ('Alta qualidade (150 dpi)',150),
             ('Maxima 4K (300 dpi)',300)],
    value=150, description='Qualidade:',
    style={'description_width':'80px'},
    layout=widgets.Layout(margin='4px 0'),
)
w_salvar = widgets.Checkbox(
    value=False, description='Salvar PNGs no disco',
    indent=False, layout=widgets.Layout(margin='8px 0'))
w_btn = widgets.Button(
    description='Gerar Plots', button_style='success',
    layout=widgets.Layout(width='150px',height='40px',margin='10px 0'))
w_out = widgets.Output()

def on_click(b):
    sel = list(w_vars.value)
    if not sel:
        with w_out: clear_output(); print('Selecione ao menos uma variavel.')
        return
    with w_out:
        clear_output(wait=True)
        print(f'Gerando {len(sel)} plot(s) -- {w_periodo.value} -- {w_dpi.value} dpi  [16:9]...')
        gerar_plots(w_periodo.value, sel, w_salvar.value, w_dpi.value)
        print('Concluido!')

w_btn.on_click(on_click)
painel = widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 10px">Plots 16:9 - Juazeiro BA - TikTok/Reels</h3>'),
    w_periodo,
    widgets.HBox([
        widgets.VBox([w_vars,
            widgets.HTML('<span style="color:#888;font-size:11px">Ctrl/Cmd+Click para multiplas.</span>')]),
        widgets.VBox([w_dpi, w_salvar, w_btn]),
    ], layout=widgets.Layout(gap='28px', align_items='flex-start')),
])
display(painel, w_out)


Output()